## MiRAGE-F: 18 维特征生成 (F-Dataset, 1 疾病 + 7 药物)

In [2]:
# --- Cell 1: F-Dataset 配置 ---
import pandas as pd
import os
# 数据路径: 自动定位项目根 (兼容从根目录或 code/ 启动)
if os.path.exists("data"):
    DATA_ROOT = "data"
elif os.path.exists("../data"):
    DATA_ROOT = "../data"
else:
    raise FileNotFoundError("找不到 data/ 目录, 请确认工作目录为项目根")
base_path = os.path.join(DATA_ROOT, "F-Dataset", "SimilarityMatrices")
mapping_base = os.path.join(DATA_ROOT, "F-Dataset", "Mapping")



# 1 个疾病相似度 (Phenotype Semantic, 与 C-Dataset 相同)
disease_sim_files = {
    'PS': 'DiseasePS.csv'
}

# 7 个药物相似度 (与 DDCD/C-Dataset 字段一致, 文件名带 _F 后缀)
drug_sim_files = {
    'Target':           'target_similarity_F.csv',
    'Category':         'category_simialrity_F.csv',
    'Conditions':       'condition_similarity_F.csv',
    'Description':      'description_similarity_F.csv',
    'Mechanism':        'mechanism_similarity_F.csv',
    'Pharmacodynamics': 'pharmacodynamics_similarity_F.csv',
    'Smile':            'SMILE_similarity_F.csv'
}

DISEASE_FEATURE_NAMES = ['PS']            # 1 维
DRUG_FEATURE_NAMES = list(drug_sim_files.keys())  # 7 维
COUNT_FEATURES = ['count_drug', 'count_disease']

q_score_cols = [f'q_score_{n}' for n in DISEASE_FEATURE_NAMES]
p_score_cols = [f'p_score_{n}' for n in DRUG_FEATURE_NAMES]
adj_q_cols   = [f'adj_q_score_{n}' for n in DISEASE_FEATURE_NAMES]
adj_p_cols   = [f'adj_p_score_{n}' for n in DRUG_FEATURE_NAMES]

FEATURE_18 = COUNT_FEATURES + q_score_cols + p_score_cols + adj_q_cols + adj_p_cols
assert len(FEATURE_18) == 18, f"Expected 18, got {len(FEATURE_18)}"
print(f"✅ 配置完成. 特征维度: {len(FEATURE_18)} (= 2计数 + 1+7原始 + 1+7交叉 = 18)")

✅ 配置完成. 特征维度: 18 (= 2计数 + 1+7原始 + 1+7交叉 = 18)


In [3]:
# --- Cell 2: 加载 1 个疾病相似度矩阵 ---
disease_sim_dict = {}
for fname, f in disease_sim_files.items():
    path = os.path.join(base_path, f)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        disease_sim_dict[fname] = df
        print(f"✅ Disease: {fname} {df.shape}  idx 示例: {list(df.index[:3])}")
    else:
        print(f"❌ 找不到 {f}")

✅ Disease: PS (313, 313)  idx 示例: ['D102100', 'D102300', 'D102400']


In [4]:
# --- Cell 3: 加载 7 个药物相似度矩阵 ---
drug_sim_dict = {}
for fname, f in drug_sim_files.items():
    path = os.path.join(base_path, f)
    if os.path.exists(path):
        df = pd.read_csv(path, index_col=0)
        drug_sim_dict[fname] = df
        print(f"✅ Drug: {fname} {df.shape}  idx 示例: {list(df.index[:3])}")
    else:
        print(f"❌ 找不到 {f}")

✅ Drug: Target (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Category (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Conditions (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Description (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Mechanism (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Pharmacodynamics (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']
✅ Drug: Smile (592, 592)  idx 示例: ['DB00007', 'DB00010', 'DB00014']


In [5]:
# --- Cell 4: 加载映射 (字符串 ID, F-Dataset 与相似度矩阵索引一致) ---
# mapping80_F: drug_ID (DrugBank 字符串), disease_ID (OMIM 字符串, 如 D102100)
# mapping_F:  DrugID (DrugBank), DiseaseID (OMIM)
# F-Dataset 与相似度矩阵的索引格式**完全一致**, 无需整数 ↔ 字符串转换

mapping80_path = os.path.join(mapping_base, "mapping80_F.csv")
mapping_full_path = os.path.join(mapping_base, "mapping_F.csv")

# 训练映射 (防泄漏, 字符串 ID)
mapping_train = pd.read_csv(mapping80_path)
mapping_train.columns = ['drug_ID', 'disease_ID']
mapping_train['drug_ID'] = mapping_train['drug_ID'].astype(str).str.strip()
mapping_train['disease_ID'] = mapping_train['disease_ID'].astype(str).str.strip()

# 全量映射 (用于标签赋值)
mapping_full = pd.read_csv(mapping_full_path)
if 'DrugID' in mapping_full.columns:
    mapping_full['DrugID'] = mapping_full['DrugID'].astype(str).str.strip()
    mapping_full['DiseaseID'] = mapping_full['DiseaseID'].astype(str).str.strip()
    mapping_full = mapping_full[['DrugID', 'DiseaseID']]
else:
    mapping_full.columns = ['drug_ID', 'disease_ID']
    mapping_full['drug_ID'] = mapping_full['drug_ID'].astype(str).str.strip()
    mapping_full['disease_ID'] = mapping_full['disease_ID'].astype(str).str.strip()

print(f"✅ mapping80 (训练, 防泄漏): {len(mapping_train):,} 关联")
print(f"✅ mapping_full (全量, 标签): {len(mapping_full):,} 关联")
print(f"   DrugBank ID 示例: {mapping_train['drug_ID'].iloc[:3].tolist()}")
print(f"   Disease ID 示例: {mapping_train['disease_ID'].iloc[:3].tolist()}")

✅ mapping80 (训练, 防泄漏): 1,553 关联
✅ mapping_full (全量, 标签): 1,933 关联
   DrugBank ID 示例: ['DB00659', 'DB00284', 'DB01193']
   Disease ID 示例: ['D103780', 'D125853', 'D608622']


In [11]:
# --- Cell 5: 核心特征计算 (18 维) ---
# 论文 Eq.3: q_score = max Sim(s', s), p_score = max Sim(d', d)
# 论文 Eq.4: adj = score × |N| (交叉乘法)
# F-Dataset: 疾病侧 1 个 PS, 药物侧 7 个 (与 C-Dataset 字段一致, 文件名带 _F)

import numpy as np
from tqdm import tqdm

# 预计算邻居索引 (字符串 ID, 来自 mapping80)
drug_to_diseases_train = mapping_train.groupby('drug_ID')['disease_ID'].apply(set).to_dict()
disease_to_drugs_train = mapping_train.groupby('disease_ID')['drug_ID'].apply(set).to_dict()

# 标签索引: drug_ID 在 disease_ID 的全量已知药物集合中
disease_to_drugs_full = {}
for _, row in mapping_full.iterrows():
    disease_to_drugs_full.setdefault(row['DiseaseID'], set()).add(row['DrugID'])

all_drugs = sorted(mapping_train['drug_ID'].unique().tolist())
all_diseases = sorted(mapping_train['disease_ID'].unique().tolist())
print(f"药物数: {len(all_drugs):,} | 疾病数: {len(all_diseases):,}")
print(f"总对数: {len(all_drugs)*len(all_diseases):,}")

rows_list = []
for drug_id in tqdm(all_drugs):
    known_diseases = drug_to_diseases_train.get(drug_id, set())

    for disease_id in all_diseases:
        known_drugs = disease_to_drugs_train.get(disease_id, set())
        Ad = known_diseases - {disease_id}
        Bs = known_drugs - {drug_id}

        count_disease = len(Ad)
        count_drug = len(Bs)

        # --- 疾病侧 (1 维) ---
        q_feats = {}
        adj_q_feats = {}
        for fname, mat in disease_sim_dict.items():
            val = 0.0
            if Ad and disease_id in mat.index:
                valid = [d for d in Ad if d in mat.index]
                if valid:
                    v = mat.loc[disease_id, valid].max()
                    if not pd.isna(v):
                        val = float(v)
            q_feats[f'q_score_{fname}'] = val
            adj_q_feats[f'adj_q_score_{fname}'] = val * count_drug

        # --- 药物侧 (7 维) ---
        p_feats = {}
        adj_p_feats = {}
        for fname, mat in drug_sim_dict.items():
            val = 0.0
            if Bs and drug_id in mat.index:
                # F-Dataset: drug ID 直接是 DrugBank 字符串, 与矩阵索引一致
                valid = [d for d in Bs if d in mat.index]
                if valid:
                    v = mat.loc[drug_id, valid].max()
                    if not pd.isna(v):
                        val = float(v)
            p_feats[f'p_score_{fname}'] = val
            adj_p_feats[f'adj_p_score_{fname}'] = val * count_disease

        # --- 组装 18 维 ---
        row = {
            'drugID': drug_id,
            'diseaseID': disease_id,
            'count_drug': count_drug,
            'count_disease': count_disease,
        }
        row.update(q_feats)
        row.update(p_feats)
        row.update(adj_q_feats)
        row.update(adj_p_feats)
        row['label'] = 1 if drug_id in disease_to_drugs_full.get(disease_id, set()) else 0

        rows_list.append(row)

df_scores = pd.DataFrame(rows_list).fillna(0.0)
feature_cols = [c for c in df_scores.columns if c not in ['drugID', 'diseaseID', 'label']]
print(f"\n✅ 计算完成! {len(df_scores):,} 行, {len(feature_cols)} 维特征 (预期 18)")
assert len(feature_cols) == 18
print(f"  正样本: {(df_scores['label']==1).sum():,} | 负: {(df_scores['label']==0).sum():,}")
df_scores.head(3)

药物数: 592 | 疾病数: 313
总对数: 185,296


100%|██████████| 592/592 [04:28<00:00,  2.20it/s]



✅ 计算完成! 185,296 行, 18 维特征 (预期 18)
  正样本: 1,932 | 负: 183,364


,drugID,diseaseID,count_drug,count_disease,q_score_PS,p_score_Target,p_score_Category,p_score_Conditions,p_score_Description,p_score_Mechanism,...,p_score_Smile,adj_q_score_PS,adj_p_score_Target,adj_p_score_Category,adj_p_score_Conditions,adj_p_score_Description,adj_p_score_Mechanism,adj_p_score_Pharmacodynamics,adj_p_score_Smile,label
0,DB00007,D102100,2,7,0.20737,0.0,0.068966,0.0,0.713892,0.647832,...,0.086466,0.41474,0.0,0.482759,0.0,4.997245,4.534823,5.366894,0.605263,0
1,DB00007,D102300,10,7,0.07867,0.0,0.057692,0.0,0.805419,0.767263,...,0.105455,0.78670,0.0,0.403846,0.0,5.637936,5.370843,5.730962,0.738182,0
2,DB00007,D102400,1,7,0.16832,0.0,0.055556,0.0,0.758346,0.606851,...,0.026316,0.16832,0.0,0.388889,0.0,5.308424,4.247954,5.225405,0.184211,0


In [12]:
# --- Cell 6: 保存结果 ---
output_file = "results/MiRAGE_score_F.csv"
os.makedirs("results", exist_ok=True)
df_scores.to_csv(output_file, index=False)

print(f"✅ 结果已保存至 {output_file}")
print(f"   总行数: {len(df_scores):,}")
print(f"   总列数: {len(df_scores.columns)} (= drugID + diseaseID + label + 18特征)")
print(f"   正样本: {(df_scores['label']==1).sum():,}")
print(f"   负样本: {(df_scores['label']==0).sum():,}")
print(f"\n前 5 行预览:")
df_scores.head()

✅ 结果已保存至 results/MiRAGE_score_F.csv
   总行数: 185,296
   总列数: 21 (= drugID + diseaseID + label + 18特征)
   正样本: 1,932
   负样本: 183,364

前 5 行预览:


,drugID,diseaseID,count_drug,count_disease,q_score_PS,p_score_Target,p_score_Category,p_score_Conditions,p_score_Description,p_score_Mechanism,...,p_score_Smile,adj_q_score_PS,adj_p_score_Target,adj_p_score_Category,adj_p_score_Conditions,adj_p_score_Description,adj_p_score_Mechanism,adj_p_score_Pharmacodynamics,adj_p_score_Smile,label
0,DB00007,D102100,2,7,0.207370,0.0,0.068966,0.0,0.713892,0.647832,...,0.086466,0.414740,0.0,0.482759,0.0,4.997245,4.534823,5.366894,0.605263,0
1,DB00007,D102300,10,7,0.078670,0.0,0.057692,0.0,0.805419,0.767263,...,0.105455,0.786700,0.0,0.403846,0.0,5.637936,5.370843,5.730962,0.738182,0
2,DB00007,D102400,1,7,0.168320,0.0,0.055556,0.0,0.758346,0.606851,...,0.026316,0.168320,0.0,0.388889,0.0,5.308424,4.247954,5.225405,0.184211,0
3,DB00007,D102500,7,7,0.213900,0.0,0.132353,0.5,0.871304,0.873573,...,0.103448,1.497300,0.0,0.926471,3.5,6.099125,6.115009,5.749167,0.724138,0
4,DB00007,D103100,1,7,0.092941,0.0,0.046875,0.0,0.762618,0.589292,...,0.101887,0.092941,0.0,0.328125,0.0,5.338326,4.125045,4.354001,0.713208,0
